# Experiment 15 — Negative controls (non-linear policies)

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Show that policies known a-priori to be non-linear — grim-trigger
(sharp threshold) and corridor (piecewise) — are NOT well-captured by the
linear policy class. This serves as a negative control.

**Data requirement.** New data with two additional policies. Prompt drafts
for both are available in the earlier v4 notebook; re-run the data
generation loop with these to produce

```
../data/0421_redo_previous_exp/results/exp06_grim_trigger.json
../data/0421_redo_previous_exp/results/exp07_corridor.json
```

The notebook will evaluate the selected linear model class on these
policies and report the MAE gap vs the main-paper policies.

In [ ]:
extra_files = ['exp06_grim_trigger.json', 'exp07_corridor.json']
found = [f for f in extra_files if (nu.RAND_BUYER_DIR / f).exists()]
if not found:
    print('Negative-control datasets not found; skipping.')
else:
    # Load and analyze each negative control
    import json
    for fn in found:
        policy_name = fn.replace('.json','').split('_', 1)[1].upper()
        fm = {policy_name: fn}
        raw_nc = nu.load_policy_runs(policy_name, data_dir=nu.RAND_BUYER_DIR, file_map=fm)
        _, turns_nc = nu.preprocess_turns(raw_nc)
        # Fit feature II + Ridge under v7 pipeline
        meta, X, y, _ = nu.build_design_matrix(turns_nc, policy_name, 'II')
        pred, _ = nu.grouped_cv_predict(meta, X, y, 'Ridge', n_splits=nu.OUTER_FOLDS)
        m = nu.cv_metrics_full(pred)
        print(f'{policy_name}: Feature II / Ridge MAE={m["mae_norm"]:.4f}, R2={m["r2"]:.3f}')